# Columnar performance

**P1 Core · D2 Independent · 100 minutes**

In [ ]:
import tempfile
from pathlib import Path
import duckdb
import pandas as pd
from pathlib import Path

def locate(relative: str, local_name: str | None = None) -> Path:
    candidates = []
    if local_name:
        candidates.append(Path.cwd() / local_name)
    candidates.extend(root / relative for root in [Path.cwd(), *Path.cwd().parents])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Cannot find {relative}. Run from the course clone or place the downloaded data beside this notebook."
    )


In [ ]:
path = locate('datasets/teaching/air-quality/observations.csv', 'observations.csv')
source = pd.read_csv(path)
expanded = pd.concat([source.assign(replicate=i) for i in range(100)], ignore_index=True)
expanded['observation_id'] = range(len(expanded))
expanded.shape

## Task

Write CSV/Parquet, use projected Parquet reads and DuckDB filtered aggregation, and verify result equality. Do not assert that one noisy timing must be faster.

In [ ]:
def columnar_evidence(frame: pd.DataFrame) -> dict:
    with tempfile.TemporaryDirectory() as directory:
        directory = Path(directory)
        csv_path = directory/'air.csv'; parquet_path = directory/'air.parquet'
        frame.to_csv(csv_path, index=False); frame.to_parquet(parquet_path, index=False)
        projected = pd.read_parquet(parquet_path, columns=['station','PM2.5','replicate'])
        pandas_result = (projected.query("station == 'Dingling'")
                         .groupby('replicate')['PM2.5'].mean().sort_index())
        query = "SELECT replicate, avg(\"PM2.5\") AS mean_pm25 FROM read_parquet(?)                 WHERE station='Dingling' GROUP BY replicate ORDER BY replicate"
        duck = duckdb.execute(query, [str(parquet_path)]).fetchdf().set_index('replicate')['mean_pm25']
        return {'rows': len(frame), 'csv_bytes': csv_path.stat().st_size,
                'parquet_bytes': parquet_path.stat().st_size,
                'selected_columns': list(projected.columns),
                'equal': bool(pd.Series(pandas_result).round(10).equals(duck.round(10)))}

In [ ]:
evidence = columnar_evidence(expanded)
assert evidence['rows'] == 100_800
assert evidence['parquet_bytes'] < evidence['csv_bytes']
assert evidence['selected_columns'] == ['station','PM2.5','replicate']
assert evidence['equal']
evidence

## Transfer

Under a fixed memory budget, compare full CSV materialisation, projected Parquet, and a filtered aggregate. Explain chunking and why distribution overhead may dominate.